# Phase 1 — C-MAPSS EDA
Run after `python -m pmaint.data.download`. Findings are summarised in `docs/phases/PHASE_01_data_eda.md`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pmaint.data.loader import SENSOR_COLS, SETTING_COLS, SUBSETS, add_train_rul, load_subset

pd.set_option('display.width', 160)

## 1. Subset overview

In [ ]:
rows = []
for s in SUBSETS:
    tr, te, r = load_subset(s)
    life = tr.groupby('unit').cycle.max()
    rows.append(dict(subset=s, train_rows=len(tr), test_rows=len(te), train_units=tr.unit.nunique(),
                     test_units=te.unit.nunique(), life_min=life.min(), life_mean=life.mean(), life_max=life.max(),
                     op_conditions=tr[SETTING_COLS[:2]].round(1).drop_duplicates().shape[0]))
pd.DataFrame(rows).set_index('subset')

## 2. FD001 — engine lifetimes

In [ ]:
train, test, rul_test = load_subset('FD001')
train = add_train_rul(train)
train.groupby('unit').cycle.max().plot.hist(bins=20, title='Engine life (cycles), FD001 train')
plt.xlabel('cycles'); plt.show()

## 3. Constant / uninformative columns

In [ ]:
desc = train[SETTING_COLS + SENSOR_COLS].describe().T
desc['constant'] = desc['std'] < 1e-6
desc[['mean','std','min','max','constant']]

## 4. Sensor trajectories vs RUL (a few engines)

In [ ]:
informative = [c for c in SENSOR_COLS if train[c].std() > 1e-6]
fig, axes = plt.subplots(5, 3, figsize=(14, 14), sharex=True)
for ax, c in zip(axes.ravel(), informative, strict=True):
    for u in (1, 25, 50, 75, 100):
        d = train[train.unit == u]
        ax.plot(d.rul, d[c], lw=.8)
    ax.set_title(c); ax.invert_xaxis()
fig.supxlabel('RUL (cycles to failure)'); plt.tight_layout(); plt.show()

## 5. Correlation of sensors with RUL

In [ ]:
train[informative + ['rul']].corr()['rul'].drop('rul').sort_values().plot.barh(figsize=(6,5), title='Pearson corr with RUL')
plt.show()

## 6. Test set: truncation and true RUL

In [ ]:
last = test.groupby('unit').cycle.max()
print('test cycles available per unit:', last.describe()[['min','mean','max']].round(1).to_dict())
rul_test.plot.hist(bins=20, title='True RUL at last test cycle, FD001'); plt.show()

## 7. Operating conditions (FD001 vs FD002)

In [ ]:
_, tr2, _ = load_subset('FD002')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].scatter(train.setting_1, train.setting_2, s=2); ax[0].set_title('FD001 settings')
ax[1].scatter(tr2.setting_1, tr2.setting_2, s=2); ax[1].set_title('FD002 settings (6 regimes)')
plt.show()